# Generate dataset


## Check paths


In [15]:
# paths

from pathlib import Path


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / "fitness-assistant").exists():
    PROJECT_DIR = PROJECT_DIR / "fitness-assistant"
elif (PROJECT_DIR / "07-project-example" / "fitness-assistant").exists():
    PROJECT_DIR = PROJECT_DIR / "07-project-example" / "fitness-assistant"

COURSE_ROOT = PROJECT_DIR.parents[1]
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Data dir:", DATA_DIR)


Project dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant
Data dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant/data


## Imports


In [16]:
# Imports and client

import os
from typing import List

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


load_dotenv(COURSE_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Add it to the course root .env file.")

openai_client = OpenAI()
MODEL = "gpt-5.4-mini"

MODEL

'gpt-5.4-mini'

## Schema


In [17]:
# Exercise schema

class Exercise(BaseModel):
    id: str = Field(description="Unique identifier, e.g. 'push-up-001'")
    exercise_name: str = Field(description="Name of the exercise")
    type_of_activity: str = Field(description="Strength, Cardio, Flexibility, etc.")
    type_of_equipment: str = Field(description="Dumbbells, Barbell, None (bodyweight), etc.")
    body_part: str = Field(description="Chest, Back, Legs, etc.")
    type: str = Field(description="Compound, Isolation, etc.")
    muscle_groups_activated: str = Field(description="Comma-separated list, e.g. 'Chest, Triceps, Shoulders'")
    instructions: str = Field(description="Detailed step-by-step instructions")


class ExerciseDataset(BaseModel):
    exercises: List[Exercise]

## Generate records


In [18]:
# Generate data

prompt = """
Generate a dataset of 50 diverse fitness exercises.
Cover different muscle groups, equipment types, and difficulty levels.
Include bodyweight exercises, free weights, and machine exercises.
""".strip()

response = openai_client.responses.parse(
    model=MODEL,
    input=[{"role": "user", "content": prompt}],
    text_format=ExerciseDataset,
)

dataset = response.output_parsed
df_raw = pd.DataFrame([exercise.model_dump() for exercise in dataset.exercises])

print(f"Generated {len(df_raw)} exercises")
df_raw.head()

Generated 50 exercises


,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a high plank with hands slightly wide...
1,squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with feet shoulder-width apart and toes ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Shoulders, Quadriceps, Calves, Glutes, Core",Stand upright with feet together and arms at y...


## Clean data


In [19]:
# Clean data

required_columns = [
    "id",
    "exercise_name",
    "type_of_activity",
    "type_of_equipment",
    "body_part",
    "type",
    "muscle_groups_activated",
    "instructions",
]

df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df[required_columns]

df["id"] = df["id"].str.strip().str.lower().str.replace(" ", "-", regex=False)
df["exercise_name"] = df["exercise_name"].str.strip()

df = df.drop_duplicates(subset=["id"])
df = df.drop_duplicates(subset=["exercise_name"])
df = df.reset_index(drop=True)

print("Rows after cleanup:", len(df))
df.head()

Rows after cleanup: 50


,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a high plank with hands slightly wide...
1,squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with feet shoulder-width apart and toes ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Shoulders, Quadriceps, Calves, Glutes, Core",Stand upright with feet together and arms at y...


## Save data


In [20]:
# save files

csv_path = DATA_DIR / "data.csv"

df.to_csv(csv_path, index=False)

print("Saved CSV:", csv_path.relative_to(PROJECT_DIR))


Saved CSV: data/data.csv


## Check saved data


In [21]:
# Load saved data

df_check = pd.read_csv(csv_path)
documents = df_check.to_dict(orient="records")

print("Dataset rows:", len(df_check))
print("Document records:", len(documents))

df_check.head()

Dataset rows: 50
Document records: 50


,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a high plank with hands slightly wide...
1,squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with feet shoulder-width apart and toes ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Shoulders, Quadriceps, Calves, Glutes, Core",Stand upright with feet together and arms at y...
